# Hare and Hound exercise: Detecting an Earth-like planet

This notebook shows how Fig. 18 of the PlatoSim3 paper was created. Note that this notebook both downloads and process the data generated with the `PLATOnium` toolkit. The original simulations were simulated on the super computing facility VSC. Be patient since several of the modules below takes some time to compute (see notes through notebook). 

### Setup notebook 

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

### Imports

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

from matplotlib import pyplot as plt
import matplotlib.colors  as colors
import matplotlib.patches as patches

from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
from PyAstronomy import pyasl

import scipy.constants as c
from scipy.signal import periodogram
from scipy.ndimage import median_filter
from scipy.interpolate import interp2d
from scipy.stats import binned_statistic

import wotan
import transitleastsquares as tls

# PlatoSim
import platosim.referenceFrames as rf
import platosim.plot            as pt
import platosim.utilities       as ut
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_paper
setup_paper()

---
## 1.1 - Fetch simulations
---

In [ ]:
# Change the "outputDir" directory below if you want to change the default
odir = os.getcwd()
fdir = 'simulations_PlatoSimPaper2023/hareAndHound'
idir = f'{odir}/{fdir}'

In [ ]:
# # Download all files from KUL FTP
# fdir = 'simulations_PlatoSimPaper2023/hareAndHound'
# ut.downloadFromFTP(fdir, odir, server='platodata')

In [ ]:
# # Download all files from KUL FTP
# fdir = 'simulations_PlatoSimPaper2023/varsource'
# ut.downloadFromFTP(fdir, odir, server='platodata')

In [ ]:
# Load all data into a lightcurve object
lcs = LightCurve(idir, mode="multi")

In [ ]:
# Unzip all compressed files
# lcs.unpack()

---
## 1.2 - Input variability model
---

In [ ]:
# Fetch the first light curve for this star
lc = LightCurve(lcs.files("ftr")[0])

In [ ]:
# Calculate the PLATO passband magnitude of G2V star
ut.passbandConversionV2P(10, 5777, inverse=True)

In [ ]:
# Get parameter used in input model
varinfo = lc.varsource_info()
P    = varinfo.P_day[0]
t0   = varinfo.t0_day[0]
tdur = wotan.t14(R_s=1, M_s=1, P=P, small_planet=True)
print(f'Orbital period     : {P:.2f} days')
print(f'Time of empherimes : {t0:.2f} days')
print(f'Transit duration   : {tdur*24:.2f} hours')

In [ ]:
# Show the noise-less light curve used
df_var = lc.varsource()
# Store varibale paraters
time_var = df_var.time / c.day
flux_var = df_var.comb
flux_tra = df_var.tran
# Print components
df_var.head()

In [ ]:
# Plot the input noise-less light curve
fig, ax = lc.plot_varsource();

In [ ]:
# Compute PSD of granulation and oscillations
freq_gran, psd_gran = periodogram(df_var.gran, 1/25, scaling='density')
freq_puls, psd_puls = periodogram(df_var.puls, 1/25, scaling='density')
freq_gran *= 1e6  # [muHz]
freq_puls *= 1e6  # [muHz]

In [ ]:
# Creaet plot
fig, ax = plt.subplots(2, 1, figsize=(9,8))
# Plot global model
ax[0].plot(freq_gran, psd_gran, "-", c='royalblue', lw=0.3, label="Granulation")
ax[0].plot(freq_puls, psd_puls, "-", c='orange', lw=0.3, label="Pulsations")
ax[0].set_xlim(1e2, np.max(freq_gran))
ax[0].set_ylim(1e0, 1e8)
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlabel(r"Frequency, $\nu$ [$\mu$Hz]")
ax[0].set_ylabel(r"PSD [ppm$^2$ $\mu$Hz$^{-1}$]")
ax[0].legend(ncol=1, loc='upper right', fontsize=16)
# Plot zoom in on p-modes
ax[1].plot(freq_puls, psd_gran+psd_puls, "-", c='k', lw=0.3, label="Stochastic oscillations")
ax[1].set_xlim(2600, 3600)
ax[1].set_ylim(1e5, 0.5e8)
ax[1].set_yscale('log')
ax[1].set_xlabel(r"Frequency, $\nu$ [$\mu$Hz]")
ax[1].set_ylabel(r"PSD [ppm$^2$ $\mu$Hz$^{-1}$]")
ax[1].legend(ncol=1, loc='upper right', fontsize=16)
plt.tight_layout()
# Save figure
fig.savefig('oscillations.png', bbox_inches='tight', dpi=200)

---
## 1.3 - Quick-look for a single camera and quarter
---

In [ ]:
# Fetch the first light curve for this star
lc = LightCurve(lcs.files("ftr")[0])

In [ ]:
# Show the data product
lc.data().head()

### Detrending with **Wotan**

The software Wotan is implemented into the detrending module of the `LightCurve` class. We use the robust fitting method cosine description. Note that the detrending works best if you in priori known the full transit time of the planet (i.e. from contact point 1 to 4). We here calculate this period and use is a `window_lenght` detrend criteria:

In [ ]:
df = lc.detrend(model="wotan", window=3*tdur, mask=[P, tdur, t0], plot=True);

### Outlier rejection with **Scipy**

In [ ]:
df = lc.clip(model='scipy', low=4, high=4, plot=True);

---
## 1.4 - Vetting for Earth-like transit
---

### Merge into single light curve

Note that commented code-block below shows how to call the function `reduce_star()`, which will first detrend and sigma-clip each mission-quarter light curve individually, and next combine all the light curves into one array and lastly average flux measurements from the same camera group (as they share identical time stamps). The script below takes a few hours to finish (hence why we have commented it out).

In [ ]:
# lc = lcs.reduce_star(ofile=f'{idir}/../lc_final_tot.ftr', flux_group_mean=True,
#                     model_detrend='wotan', window=3*tdur, mask=[P, tdur, t0],
#                     model_clip='scipy', low=4, high=4)

Alternatively you can simply download the final light curve prepared for vetting using the above script:

In [ ]:
filename = 'simulations_PlatoSimPaper2023/lc_final.ftr'
ut.downloadFromFTP(filename,  os.getcwd(), server='platodata')

In [ ]:
# Load the final light curve
lc = LightCurve(f'{idir}/../lc_final.ftr')
df = lc.data()

In [ ]:
# Remove small section where the detrending did a bad job
df = df.drop(df[(df.time > 338*c.day) & (df.time < 340*c.day)].index)
time = df.time / c.day
flux = df.flux * 1e6

In [ ]:
# Mask transits using Wotan
mask = wotan.transit_mask(time=time, period=P, duration=tdur, T0=t0)
time_mask = time[mask]
flux_mask = flux[mask]

In [ ]:
# Plot the corrected timeseries
fig, ax = plt.subplots(1, 1, figsize=(9,4))
ax.plot(time,      flux,      ',', c='k', ms=1.0, alpha=0.01)
ax.plot(time_mask, flux_mask, ',', c='deeppink', ms=1.0, alpha=1)
# Labels
ax.set_xlabel(r"Time [days]")
ax.set_ylabel(r"Relative flux [ppm]")
# Settings
ax.set_xlim(np.min(time), np.max(time))
plt.tight_layout();

### Bin data per 1h

In [ ]:
# Resample (bin) data using scipy
bins = int((time.iloc[-1]-time.iloc[0])*24)
flux_bin, time_bin, nbin = binned_statistic(time, flux, statistic='median', bins=bins)
time_bin = time_bin[:-1] + np.diff(time_bin)[0]/2.

In [ ]:
# Remove extreme outliers
time_bin = time_bin[(flux_bin > -250) & (flux_bin < 250)]
flux_bin = flux_bin[(flux_bin > -250) & (flux_bin < 250)]

In [ ]:
# Plot the corrected timeseries
fig, ax = plt.subplots(1,1,figsize=(9,4))
ax.plot(time_bin, flux_bin, '.', c='deeppink', ms=10,  alpha=1.0, mec='k', label='1h bins');
ax.plot(time_var, flux_tra, '-', c='orange',   lw=2,   alpha=1.0, label='TLS fit')
# Labels
ax.set_xlabel(r"Time [days]")
ax.set_ylabel(r"Relative flux [ppm]")
# Settings
ax.set_xlim(np.min(time), np.max(time))
plt.tight_layout()

### Transit vetting with TLS

In [ ]:
# 2 transits
dex2 = np.where((time_bin > 100) & (time_bin < 1000))[0]
time_bin2 = time_bin[dex2]
flux_bin2 = flux_bin[dex2]
flux_bin2 = (flux_bin2 - np.mean(flux_bin2)) / 1e6 + 1

# 3 transits
dex3 = np.where((time_bin > 100) & (time_bin < 1300))[0]
time_bin3 = time_bin[dex3]
flux_bin3 = flux_bin[dex3]
flux_bin3 = (flux_bin3 - np.mean(flux_bin3)) / 1e6 + 1

# 4 transits (entire dataset)
time_bin4 = time_bin
flux_bin4 = flux_bin
flux_bin4 = (flux_bin4 - np.mean(flux_bin4)) / 1e6 + 1

In [ ]:
# Function to launch TLS computation 
def run_tls(time, flux): 
    model   = tls.transitleastsquares(time, flux)
    results = model.power(R_star=1.,
                      R_star_min=0.13,
                      R_star_max=3.5,
                      M_star=1.,
                      M_star_min=0.1,
                      M_star_max=1.5,
                      period_min=10., 
                      period_max=400.,
                      n_transits_min=2,
                      use_threads=10)
    return results

In [ ]:
# Perform TLS vetting
results4 = run_tls(time_bin4, flux_bin4)
results3 = run_tls(time_bin3, flux_bin3)
results2 = run_tls(time_bin2, flux_bin2)

In [ ]:
print(f'Best period  : [{results2.period:.3f}, {results3.period:.3f}, {results4.period:.3f}] days')
print(f'Best duration: [{results2.duration:.5f}, {results3.duration:.5f}, {results4.duration:.5f}] days')
print(f'Transit depth: [{results2.depth:.5f}, {results3.depth:.5f}, {results4.depth:.5f}]')
print(f'Signal detection efficiency (SDE): [{results2.SDE:.2f}, {results3.SDE:.2f}, {results4.SDE:.2f}]')
print(f'Signal to noise ratio       (SNR): [{results2.snr:.2f}, {results3.snr:.2f}, {results4.snr:.2f}]')

In [ ]:
# Plot SDE
plt.figure(figsize=(9,4))
ax = plt.gca()
ax.axvline(results4.period, alpha=0.4, lw=3)
plt.plot(results2.periods, results2.power, c='orange',    lw=1.0)
plt.plot(results3.periods, results3.power, c='royalblue', lw=1.0)
plt.plot(results4.periods, results4.power, c='k',         lw=0.5)
# Plot harmonics from transits
for n in range(2, 10):
    ax.axvline(results4.period / n, alpha=0.4, lw=1, linestyle="dashed")
# Settings
plt.ylabel(r'SDE')
plt.xlabel('Period [days]')
plt.xlim(np.min(results4.periods), np.max(results4.periods))
plt.tight_layout();

### Final plot for paper

In [ ]:
# Prepare for phase plot
phase = pyasl.foldAt(time, P, T0=t0+P/2)
sort  = np.argsort(phase)
# For variable
phase_var = pyasl.foldAt(time_var, P, T0=t0+P/2)
sort_var  = np.argsort(phase_var)
# Binned data
phase_bin = pyasl.foldAt(time_bin, P, T0=t0+P/2)
sort_bin  = np.argsort(phase_bin)
# TLS fit for 4 transits
phase_res = results4.model_folded_phase
flux_res  = (results4.model_folded_model-1)*1e6

In [ ]:
f = lcs.files(suffix='ftr', group=1, camera=1)
lc = LightCurve(f[0])
lc.data()

In [ ]:
# Fetch filenames for N-CAM 1.1 of all quarters
filenames = lcs.files(suffix='ftr', group=1, camera=1)
df0 = pd.DataFrame()
N = len(filenames)
for i,f in zip(range(1,N+1), filenames):
    print(f'Detreding light curve nr {i} out of {N}', end='\r')
    lc = LightCurve(f)
    df = lc.detrend(model="wotan", window=3*tdur, mask=[P, tdur, t0])
    df0[f'time_Q{i}'] = df.time / c.day
    df0[f'flux_Q{i}'] = df.flux_trend / 1e3

In [ ]:
# Create overview plot of transit vetting
fig, ax0 = lcs.plot_multi(group=1, camera=1, quarter=False, figsize=(9,11))
ax0.set_ylim(93, 103)
colors = ['deeppink', 'orange', 'royalblue']
# Plot Wotan detrends
import matplotlib.patheffects as mpe                                                                                                                                                          
outline = mpe.withStroke(linewidth=3, foreground='k')
for i in range(1,N+1):
    if i == 1:
        ax0.plot(df0[f'time_Q{i}'], df0[f'flux_Q{i}']/4.026526, '-', c='w', lw=0.5, path_effects=[outline], 
                 label='Wotan trend') 
    else:
        ax0.plot(df0[f'time_Q{i}'], df0[f'flux_Q{i}']/4.026526, '-', c='w', lw=1, path_effects=[outline])                                                                                                                                                                                                                                                                                                  
# Plot transit times for ax 0 and 1
ymax = 0.73
transit_times = np.arange(t0, 4*P, P)
for T in transit_times:
    ymax -= 0.10 
    ax0.axvline(x=T, ymax=ymax, c=colors[0], linestyle='--', lw=1.3)
    if T == transit_times[-1]:
        ax0.axvline(x=T, ymax=ymax, c=colors[0], linestyle='--', lw=1.3, label='Transit times')
#     ax1.axvline(x=T, ymax=0.1,  c='deeppink', linestyle='--', lw=1.3)
ax0.legend(loc='upper right', ncols=2, fontsize=15, frameon=True)
    
# Plot the corrected timeseries
ax1 = fig.add_subplot(312)
ax1.plot(time,       flux/1e3,       ',', c='k', ms=1.0, alpha=0.01)
ax1.plot(time[mask], flux[mask]/1e3, '.', c=colors[0], ms=0.1, alpha=1, label='In-transit data')
# Labels
ax1.set_xlabel(r"Time [days]")
ax1.set_ylabel(r"Relative flux [ppt]")
# Settings
leg1 = ax1.legend(loc='upper right', ncols=1, fontsize=15, frameon=True)
# leg1.legendHandles[0]._legmarker.set_alpha(1)
leg1.legendHandles[0].set_markersize(5)
ax1.set_xlim(np.min(time), np.max(time))
ax1.set_ylim(-5, 5)

# Plot the corrected timeseries
ax2 = fig.add_subplot(313)
ax2.plot(phase_var[sort_var], flux_var[sort_var], '-', c=colors[2], lw=0.5, alpha=1.0, label='Input model')
ax2.plot(phase_bin[sort_bin], flux_bin[sort_bin], '.', c=colors[0],  ms=10,  alpha=1.0, mec='k', label='1h bins')
ax2.plot(phase_res, flux_res, '-', c=colors[1], lw=3, label='TLS fit')
# Labels
ax2.set_xlabel(r"Orbital phase")
ax2.set_ylabel(r"Relative flux [ppm]")
ax2.legend(loc='upper right', ncols=3, fontsize=15, frameon=True)
# Settings
ax2.set_xlim(0.496, 0.504)
ax2.set_ylim(-250, 250)

# Save the figure
plt.tight_layout(w_pad=0.5)
fig.savefig('transit.png', bbox_inches='tight', dpi=200, transparent=False);    